In [ ]:
import json
import numpy as np
import pandas as pd
import regex
import requests
from src.utils import (
    dateFromText,
    getEstacionamientos,
    getFilesByDate,
    getFilesByWeek,
    guardarExcel,
    guardarExcelMulti,
    isEmpty,
    isValidCode,
    listTopos,
    loadEstaciones,
    localizeFecha,
    map_cod2name,
    map_name2use_name,
    parallelizeFunction,
    rellenarId,
    setEF,
    slidingWindow,
    splitDataframe,
    time2localtime,
    splitList,
)
from src.utils import (
    getEstacionamientos,
    parallelizeFunction,
    splitDataframe,
    splitList,
    time2localtime,
)
from pathlib import Path
from src.api.APIs import getInfoToposMSE


In [ ]:
def processJCTC(df_movimientos:pd.DataFrame):
    estaciones = loadEstaciones()
    data = estaciones[["CTC","Mnemónico","Nombre","Código"]].drop_duplicates()
    df_movimientos=df_movimientos.merge(data,left_on= ["ctc", "interlock"], right_on=["CTC","Mnemónico"])
    df_movimientos["stationName"] = df_movimientos[["Nombre"]]
    df_movimientos["stationCode"] = df_movimientos[["Código"]]
    df_movimientos.drop(["CTC","Mnemónico","Nombre","Código"],axis=1)
    rename_cols_jctc = {
    "datetime": "Fecha",
    "technicalNumber": "NTécnico",
    "stationName": "Nombre",
    "stationCode": "Código",
    "ctc": "CTC",
    "interlock":"Mnemónico",
    "element": "Elemento",
    "direction": "Sentido",
    #"sequence": "Secuencia",
    # "messageSource": "FuenteMensaje",
    "movementType": "Movimiento",
    # "movementSubType": "SubtipoMovimiento",
    "movementSource": "FuenteMovimiento",
    # "registryType": "Registro",
    #"platform": "Vía",
    # "platformType": "TipoVía",
    #"sourcePlatform": "FuenteVía",
    #"delayInSeconds": "Retraso (segundos)",
    #"operator": "CategoríaCirculación",
    #"product": "Producto",
    #"trainCompanyCode": "Empresa",
    # "triggerElement": "ElementoDisparo",
    #"originDate": "FechaOrigen",
    # "travelTime": "",
    # "parkingTime": "",
    # "platformCommercialCode": "",
    # "commercialLineCode": "LíneaComercial",
    # "originPlannedPointCode": "",
    # "originPlannedPointName": "",
    # "destinationPlannedPointCode": "",
    # "destinationPlannedPointName": "",
    # "technicalDeparturePlanned": "",
    # "descriptionMap": "",
    }
    map_movimientos_jctc = {
    "OCCUPATION":"LLEGADA"
    }
    df_movimientos = df_movimientos[list(rename_cols_jctc.keys())].rename(
        columns=rename_cols_jctc
    )
    df_movimientos["Movimiento"] = df_movimientos["Movimiento"].apply(
    map_movimientos_jctc.get
    )
    df_movimientos[["Fecha"]] = pd.concat(
    parallelizeFunction(
        lambda x: x.map(time2localtime, unit="ms"),
        data=splitDataframe(df_movimientos[["Fecha"]], 1000),
        show_progress=True,
        desc="Formateando fechas.",
        output="series",
        )
    )
    return df_movimientos


In [ ]:


def hacerPeticion(method: str, URL: str, headers: dict = {}, data: dict = None):

    headers = {
        "Content-Type": "application/json; charset=UTF-8",
        "Prefer": "respond-async",
        **headers,
    }

    response = requests.request(method, URL, headers=headers, data=data)
    if response.status_code == 200:
        return response
    print("")
    if response.status_code >= 400:
        print(f"Error: '{response.status_code}' en la respuesta")
        return
    if response.status_code >= 300:
        print(f"Redirección: código'{response.status_code}'")
        return
    if response.status_code > 200:
        print(f"👍: código'{response.status_code}'")
        return
    if response.status_code < 200:
        print(f"Info: código'{response.status_code}'")
        return
    if not response.encoding == "utf-8":
        response.encoding = "utf-8"
    return response

In [ ]:
def getHistoricoMOW(
    estaciones: list[str],
    trenes: list[str],
    inicio: str,
    fin: str,
    xSIV: bool = True,
    jCTC: bool = False,
    pro: bool = True,
    xREG: bool=False,
    maniobra: bool = False,
):
    """
    Obtiene el histórico de movimientos de las estaciones y/o trenes seleccionados en las fechas correspondientes

    Parámetros
    ----------
    - estaciones: list[str]
        Lista de códigos de estación
    - trenes: list[str]
        Lista de códigos técnicos de tren
    - inicio: str
        Fecha desde la que se quiere los registros en formato YYYY-mm-dd
    - fin: str
        Fecha hasta la que se quiere los registros en formato YYYY-mm-dd

    """
    HOSTPATH = "http://info.api.elcano.operaciones.adif/movementsmanager/history/"
    if not pro:
        HOSTPATH = HOSTPATH.replace(".api.", ".api.pre.")

    def iterateHistoric(trenes, ranges, inicio, fin):
        movimientos = []
        page = 0
        last_timestamp = (
            int(pd.to_datetime(inicio).tz_localize("Europe/Madrid").timestamp()) * 1000
        )
        print(last_timestamp)
        print("incio:",pd.to_datetime(inicio).tz_localize("Europe/Madrid").timestamp())
        print("fin:",pd.to_datetime(fin).tz_localize("Europe/Madrid").timestamp())
        
        while True:
            print(f"\rPágina {page} ({inicio} - {fin})", end="")
            data = {
                "page": page,
                "size": 10000,
                "initDate": int(
                    pd.to_datetime(inicio).tz_localize("Europe/Madrid").timestamp()
                )
                * 1000,
                "endDate": int(
                    pd.to_datetime(fin).tz_localize("Europe/Madrid").timestamp()
                )
                * 1000,
                "lastTimestamp": last_timestamp,
                "technicalNumbers": trenes,
                "technicalNumberRanges": ranges,
                "stationCodes": estaciones,
                "xSIV": xSIV,
                "jCTC": jCTC,
                "xREG":xREG,
            }
            data = json.dumps(data)

            response = hacerPeticion(
                "POST",
                HOSTPATH,
                # headers=headers,
                data=data,
            )
            if not response:
                break
            response = response.json()
            movimientos.extend(response["movementList"]["list"])
            page += 1
            if page == response["totalPages"]:
                break
            last_timestamp = movimientos[-1]["datetime"]
        return movimientos

    if len(trenes) >= 1000:
        movimientos = iterateHistoric([], ["00000-99999"], inicio, fin)
    elif len(trenes) > 100:
        sep_trenes = splitList(sorted(trenes), 100)
        movimientos = []
        for t in sep_trenes:
            movimientos.extend(iterateHistoric(t, [], inicio, fin))
    else:
        movimientos = iterateHistoric(trenes, [], inicio, fin)

    df_movimientos = pd.DataFrame(movimientos)
    if df_movimientos.empty:
        return df_movimientos

    return df_movimientos


In [ ]:
ntrenes = [rellenarId(el) for el in np.arange(100000)]

In [ ]:
historico = getHistoricoMOW(estaciones=[], trenes=ntrenes, inicio="2025-11-27", fin="2025-11-28", xSIV=False,jCTC=False,xREG=True)


In [ ]:
def processXreg(df:pd.DataFrame):
    rename_cols ={
        "datetime": "FechaHora",
        "technicalNumber":"NTécnico",
        "stationName":"Nombre",
        "stationCode":"Código",
        "startLocationCode":"CódigoIncio",
        "startLocationName":"NombreInicio",
        "endLocationCode":"CódigoFin",
        "endLocationName":"NombreFin",
        "ctc":"CTC",
        "timestampMSG":"TimestampMSG",
        "interlock": "Enclavamiento",
        "direction":"Dirección",
        "sequence":"Secuencia",
        "startLocationSequence":"SecuenciaInicio",
        "endLocationSequence":"SecuenciaFin",
        "messageSource":"FuenteMensaje",
        "movementType":"TipoMovimiento",
        "movementSource":"FuenteMovimiento",
        "registryType":"TipoRegistro",
        "platform":"Vía",
        "platformType":"TipoVía",
        "sourcePlatform":"FuenteVía",
        "delayInSeconds":"Retraso(s)",
        "operator":"Operador",
        "product":"Producto",
        "trainCompanyCode":"CódigoCompañia",
        "triggerElement":"ElementoTrigger",
        "element":"Elemento",
        "originDate":"FechaOrigen",
        "travelTime":"TiempoViaje",
        "parkingTime":"TiempoEstacionamiento",
        "platformCommercialCode":"CodigoVíaComercial",
        "commercialLineCode":"CódigoLineaComercial",
        "networkName":"NombreRed",
        "originPlannedPointCode":"CódigoOrigenPlanificado",
        "originPlannedPointName":"NombreOrigenPlanificado",
        "destinationPlannedPointCode":"CódigoOrigenPlanificado",
        "destinationPlannedPointName":"NombreDestinoPlanificado",
        "technicalDeparturePlanned":"SalidaTécnicaPlanificado",
        "descriptionMap":"Descripción"
        
    }
    df = df[list(rename_cols.keys())].rename(
        columns=rename_cols
    )
    df[["FechaHora"]] = pd.concat(
        parallelizeFunction(
            lambda x: x.map(time2localtime, unit="ms"),
            data=splitDataframe(historico[["FechaHora"]], 1000),
            show_progress=True,
            desc="Formateando fechas.",
            output="series",
            )
    )
    return df

In [ ]:
historico = historico[list(rename_cols.keys())].rename(
    columns=rename_cols
)

In [ ]:
    historico[["FechaHora"]] = pd.concat(
    parallelizeFunction(
        lambda x: x.map(time2localtime, unit="ms"),
        data=splitDataframe(historico[["FechaHora"]], 1000),
        show_progress=True,
        desc="Formateando fechas.",
        output="series",
        )
    )

In [ ]:
historico

In [ ]:
df = processXsiv(historico, estaciones=[])
df

In [ ]:
rename_cols = {
        "datetime": "Fecha",
        "technicalNumber": "NTécnico",
        "stationName": "Nombre",
        "stationCode": "Código",
        # "ctc": "CTC",
        "sequence": "Secuencia",
        # "messageSource": "FuenteMensaje",
        "movementType": "Movimiento",
        # "movementSubType": "SubtipoMovimiento",
        "movementSource": "FuenteMovimiento",
        # "registryType": "Registro",
        "platform": "Vía",
        # "platformType": "TipoVía",
        "sourcePlatform": "FuenteVía",
        "delayInSeconds": "Retraso (segundos)",
        "operator": "CategoríaCirculación",
        "product": "Producto",
        "trainCompanyCode": "Empresa",
        # "triggerElement": "ElementoDisparo",
        # "element": "Elemento",
        "originDate": "FechaOrigen",
        # "travelTime": "",
        # "parkingTime": "",
        # "platformCommercialCode": "",
        # "commercialLineCode": "LíneaComercial",
        # "originPlannedPointCode": "",
        # "originPlannedPointName": "",
        # "destinationPlannedPointCode": "",
        # "destinationPlannedPointName": "",
        # "technicalDeparturePlanned": "",
        # "descriptionMap": "",
    }

In [ ]:
historico = historico[list(rename_cols.keys())].rename(
    columns=rename_cols
)


In [ ]:
from src.utils import listTopos
estaciones = loadEstaciones()
estaciones = (
        estaciones.loc[
            np.invert(estaciones[["CTC", "Código"]].map(isEmpty).any(axis=1)),
            ["CTC", "Código"],
        ]
        .groupby(["Código"])
        .agg(lambda x: ",".join(set(x)))
        .reset_index()
    )
estacionamientos = getEstacionamientos(historico["Código"].unique().tolist())[
        ["Código", "TipoVía", "Vía"]
    ].drop_duplicates()
estacionamientos

In [ ]:
import json
from collections import Counter
from pathlib import Path
from typing import Union

import numpy as np
import pandas as pd
import regex


TOPOS = Path(r"C:\topogen-adif-repo\baseline")


def listTopos(ftype: str = "properties", estaciones: list[str] = []):
    """
    Dataframe con los códigos y ficheros `properties` o `pl`.\\
    `estaciones` es una lista de códigos de estación. Si está vacía, incluye todas.
    """
    list_topos_all = pd.DataFrame(TOPOS.rglob(f"*.{ftype}"), columns=[ftype])
    # list_topos_all["Código"] = list_topos_all[ftype].apply(
    #     lambda x: regex.search(r"\b[\w\d]\d{4}\b", str(x))
    # )
    # list_topos_all["Código"] = list_topos_all["Código"].apply(
    #     lambda x: x.group() if x else None
    # )
    # list_topos_all = list_topos_all.groupby(by=["Código"]).agg(set).reset_index()
    # if estaciones:
    #     list_topos_all = list_topos_all[list_topos_all["Código"].isin(estaciones)]
    return list_topos_all


In [ ]:
list_topos_all = listTopos("pl", historico["Código"].unique().tolist())
ftype: str = "properties"
list_topos_all = pd.DataFrame(TOPOS.rglob(f"*.{ftype}"), columns=[ftype])
list_topos_all

In [ ]:
historico["descriptionMap"].head(1)

In [ ]:
historico.loc(historico["movementSubType"] == "LIBERATION")

In [ ]:
df_movimientos_jctc = historico.loc[historico["messageSource"] == "JCTC"]
df_movimientos_xsiv = historico.loc[historico["messageSource"] == "XSIV"]

In [ ]:
df_movimientos_xsiv.head(2)

In [ ]:
df_movimientos_xsiv = processXsiv(df_movimientos_xsiv,estaciones=[])

In [ ]:
df_movimientos_xsiv.head(2)

In [ ]:
df_movimientos_jctc = processJCTC(df_movimientos_jctc)

In [ ]:
df_movimientos_jctc.head(2)

In [ ]:
rename_cols = {
        "datetime": "Fecha",
        "technicalNumber": "NTécnico",
        "stationName": "Nombre",
        "stationCode": "Código",
        "ctc": "CTC",
        "interlock": "Mnemónico",
        "element": "Elemento",
        "direction": "Sentido",
        "sequence": "Secuencia",
        "messageSource": "FuenteMensaje",
        "movementType": "Movimiento",
        # "movementSubType": "SubtipoMovimiento",
        "movementSource": "FuenteMovimiento",
        # "registryType": "Registro",
        "platform": "Vía",
        # "platformType": "TipoVía",
        "sourcePlatform": "FuenteVía",
        "delayInSeconds": "Retraso (segundos)",
        "operator": "CategoríaCirculación",
        "product": "Producto",
        "trainCompanyCode": "Empresa",
        # "triggerElement": "ElementoDisparo",
        "originDate": "FechaOrigen",
        # "travelTime": "",
        # "parkingTime": "",
        # "platformCommercialCode": "",
        # "commercialLineCode": "LíneaComercial",
        # "originPlannedPointCode": "",
        # "originPlannedPointName": "",
        # "destinationPlannedPointCode": "",
        # "destinationPlannedPointName": "",
        # "technicalDeparturePlanned": "",
        # "descriptionMap": "",
    }
historico = historico[list(rename_cols.keys())].rename(
        columns=rename_cols
    )
historico.head(2)

In [ ]:
historico[["Fecha", "FechaOrigen"]] = pd.concat(
         parallelizeFunction(
             lambda x: x.map(time2localtime, unit="ms"),
             data=splitDataframe(historico[["Fecha", "FechaOrigen"]], 1000),
             show_progress=True,
             desc="Formateando fechas.",
             output="series",
         )
     )

In [ ]:
historico.head(2)

In [ ]:
historico.loc[historico["FuenteMensaje"] == "JCTC", ["Fecha","NTécnico","Nombre","Código","CTC","Mnemónico","Elemento","Sentido","Movimiento","FuenteMovimiento"]] = df_movimientos_jctc[["Fecha","NTécnico","Nombre","Código","CTC","Mnemónico","Elemento","Sentido","Movimiento","FuenteMovimiento"]].iloc[:historico.loc[historico["FuenteMensaje"] == "JCTC"].shape[0]].values

In [ ]:
historico.loc[historico["FuenteMensaje"] == "XSIV", ['Fecha','NTécnico','Nombre','Código','Secuencia','Movimiento','FuenteMovimiento','Vía','FuenteVía','Retraso (segundos)','CategoríaCirculación','Producto','Empresa','FechaOrigen'	,'TipoVía']] = df_movimientos_xsiv[['Fecha','NTécnico','Nombre','Código','Secuencia','Movimiento','FuenteMovimiento','Vía','FuenteVía','Retraso (segundos)','CategoríaCirculación','Producto','Empresa','FechaOrigen','TipoVía']].iloc[:historico.loc[historico["FuenteMensaje"] == "XSIV"].shape[0]].values

In [ ]:
historico

In [ ]:
rename_cols = {
        "datetime": "Fecha",
        "technicalNumber": "NTécnico",
        "stationName": "Nombre",
        "stationCode": "Código",
        "ctc": "CTC",
        "interlock": "Mnemónico",
        "element": "Elemento",
        "direction": "Sentido",
        "sequence": "Secuencia",
        "messageSource": "FuenteMensaje",
        "movementType": "Movimiento",
        # "movementSubType": "SubtipoMovimiento",
        "movementSource": "FuenteMovimiento",
        # "registryType": "Registro",
        "platform": "Vía",
        # "platformType": "TipoVía",
        "sourcePlatform": "FuenteVía",
        "delayInSeconds": "Retraso (segundos)",
        "operator": "CategoríaCirculación",
        "product": "Producto",
        "trainCompanyCode": "Empresa",
        # "triggerElement": "ElementoDisparo",
        "originDate": "FechaOrigen",
        # "travelTime": "",
        # "parkingTime": "",
        # "platformCommercialCode": "",
        # "commercialLineCode": "LíneaComercial",
        # "originPlannedPointCode": "",
        # "originPlannedPointName": "",
        # "destinationPlannedPointCode": "",
        # "destinationPlannedPointName": "",
        # "technicalDeparturePlanned": "",
        # "descriptionMap": "",
    }
historico = historico[list(rename_cols.keys())].rename(
        columns=rename_cols
    )


In [ ]:
historico = pd.merge(
    historico,
    estacionamientos[["Código", "TipoVía", "Vía"]].drop_duplicates(),
    how="left",
    on=["Código", "Vía"],
)


In [ ]:
from src.api.APIs import getHistoricoMOW

In [ ]:
historico = getHistoricoMOW(
        estaciones=[], trenes=["10768"], inicio="2025-05-07", fin="2025-05-08", xSIV=False,JCTC=True)

In [ ]:
historico.head(5)